# Direction 1 — Direct Ordinal-Geometry Text Embeddings
## DeepDRiD Benchmark — Training, Baselines, Ablations & Figures

Companion to `DeepDRiD_FewShot_DR_fixed.ipynb` for the `direction1-ordinal-geometry` branch.
Instead of learning prompt tokens, Direction 1 learns the five grade embeddings directly in
CLIP's joint space, starting from clinical text, and shapes them with three geometry losses
(ladder, path, anchor). See `docs/superpowers/specs/` for the design.

Every run is a config file under `configs/` plus `data.k_shot` / `data.seed`. Runs are cached in the
shared `results_shared/` folder, keyed by their resolved config, so a model trained in the main notebook
or in another direction notebook is reused here, not retrained. Tables and figures go to `results_d1/`.

| Section | What it does |
|---|---|
| 1–3 | setup, experiment grid, data check |
| 4 | single-run helper (shared run cache) |
| 5 | **Phase 1**: Direction 1 only, then compare the main models of master and Directions 1–3 |
| 6 | **Phase 2** (off by default): baselines and Direction 1 ablations |
| 7 | mean ± std tables |
| 8 | figures: k-shot curves, ablation bars, confusion matrices, error decay, embedding geometry |

## 1 · Environment Setup

In [ ]:
import os, sys, io, json, warnings, logging, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm   # widget bars in Jupyter, text bars in a terminal

warnings.filterwarnings('ignore')
for _name in ('pytorch_lightning', 'lightning', 'lightning.pytorch', 'lightning_fabric'):
    logging.getLogger(_name).setLevel(logging.ERROR)

# ── Locate project root (contains 'configs/' dir) ────────────────────────────
ROOT = Path.cwd()
while not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.utils.experiment import (find_shared_results, find_runs, cached_metrics,
                                  load_run_config, run_dir, run_experiment as _run_experiment)

# ── Data folder ──────────────────────────────────────────────────────────────
# data/ is not in git. A worktree has none, so look in the project root and its
# parents (the main checkout), or set DATA_DIR yourself.
DATA_DIR = None
if DATA_DIR is None:
    for base in [ROOT, *ROOT.parents]:
        if (base / 'data/DeepDRiD/CFP_manifest.csv').exists():
            DATA_DIR = base / 'data/DeepDRiD'
            break
assert DATA_DIR is not None, 'DeepDRiD manifest not found: run scripts/prepare_external_datasets.py and src/data/build_splits.py in the main checkout, or set DATA_DIR.'
MANIFEST, SPLITS_DIR = DATA_DIR / 'CFP_manifest.csv', DATA_DIR / 'splits'

RESULTS = ROOT / 'results_d1'                   # tables and figures of this notebook
SHARED_RESULTS = find_shared_results(DATA_DIR)   # runs, shared with every notebook and branch
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root : {ROOT}')
print(f'Data         : {DATA_DIR}')
print(f'Results dir  : {RESULTS}')
print(f'Shared runs  : {SHARED_RESULTS}')
print(f'Device       : {DEVICE} | PyTorch {torch.__version__} | PL {pl.__version__}')

## 2 · Experiment Grid

Each entry maps a name to a config file. `own_budget=True` keeps the config's own epochs and
schedule (the official OrdinalCLIP / CLIP-DR recipes and native CLAP); every other run uses `EPOCHS`,
the same budget as the main notebook. Use the same names in every notebook: the comparison of main
models finds runs by name.

In [ ]:
K_SHOTS = [-1, 0, 1, 2, 4, 8, 16, 32]   # -1 = full data, 0 = zero-shot (no training)
SEEDS   = [1, 2, 3]
EPOCHS  = 50

# This notebook's main model (Phase 1)
MAIN = 'Direction 1'

# Main comparison: Direction 1 against our master model and the matched baselines
METHODS = {
    'Direction 1':            {'config': 'configs/direction1/d1_full.yaml'},
    'Ours (master)':          {'config': 'configs/deepdrid_cfp.yaml'},
    'OrdinalCLIP (matched)':  {'config': 'configs/baselines/ordinalclip_matched.yaml'},
    'CLIP-DR (matched)':      {'config': 'configs/baselines/clipdr_matched.yaml'},
    'L2RCLIP (our impl.)':    {'config': 'configs/baselines/l2rclip_ours_impl.yaml'},
    'CLAP (matched)':         {'config': 'configs/baselines/clap_matched.yaml'},
    'OrdinalCLIP (official)': {'config': 'configs/baselines/ordinalclip_official.yaml', 'own_budget': True},
    'CLIP-DR (official)':     {'config': 'configs/baselines/clipdr_official.yaml', 'own_budget': True},
    'CLAP (native)':          {'config': 'configs/baselines/clap_native.yaml', 'own_budget': True},
}

# Direction 1 ablations (each differs from d1_full in one thing)
ABLATIONS = {
    'Direction 1':                {'config': 'configs/direction1/d1_full.yaml'},
    'w/o ladder':                 {'config': 'configs/direction1/d1_no_ladder.yaml'},
    'w/o path':                   {'config': 'configs/direction1/d1_no_path.yaml'},
    'w/o anchor':                 {'config': 'configs/direction1/d1_no_anchor.yaml'},
    'w/o geometry losses':        {'config': 'configs/direction1/d1_no_geometry.yaml'},
    'CE + geometry only':         {'config': 'configs/direction1/d1_ce_geometry.yaml'},
    'init: plain prompts':        {'config': 'configs/direction1/d1_init_plain.yaml'},
    'init: random orthogonal':    {'config': 'configs/direction1/d1_init_random.yaml'},
    'anchor: inverse-sqrt freq.': {'config': 'configs/direction1/d1_anchor_invsqrt.yaml'},
    '+ pairwise loss':            {'config': 'configs/direction1/d1_plus_pairwise.yaml'},
}
ABLATION_K_SHOTS = [-1, 1, 4, 8, 16, 32]

n_main = len(METHODS) * len(K_SHOTS) * len(SEEDS)
n_abl = (len(ABLATIONS) - 1) * len(ABLATION_K_SHOTS) * len(SEEDS)
print(f'Phase 1: {len(K_SHOTS) * len(SEEDS)} runs of {MAIN} | Phase 2: {n_main} method runs + {n_abl} ablation runs '
      '(anything already in the shared cache is reused)')

## 3 · Data Check

The manifest uses per-eye grades; every k-shot split must exist before training.

In [ ]:
manifest = pd.read_csv(MANIFEST)
print(f'{len(manifest):,} images | ' + ', '.join(f"{s}: {n:,}" for s, n in manifest['split'].value_counts().items()))
missing = [(k, s) for k in K_SHOTS if k > 0 for s in SEEDS
           if not (SPLITS_DIR / f'cfp_{k}shot_s{s}_support.txt').exists()]
assert not missing, f'Missing split files for (k, seed) = {missing}: run src/data/build_splits.py'
print('All split files present.')

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})
GRADE_NAMES = ['Grade 0\n(No DR)', 'Grade 1\n(Mild)', 'Grade 2\n(Moderate)',
               'Grade 3\n(Severe)', 'Grade 4\n(Proliferative)']
fig, ax = plt.subplots(figsize=(6.4, 3.9), constrained_layout=True)
grades = np.arange(5)
for i, (split, label, color) in enumerate([('train', 'Train', '#4361EE'), ('test', 'Test', '#D97706')]):
    counts = manifest[manifest['split'] == split]['grade'].value_counts().reindex(grades, fill_value=0)
    bars = ax.bar(grades + (i - 0.5) * 0.40, counts.values, width=0.38, color=color, edgecolor='white',
                  label=f'{label} (n = {counts.sum():,})')
    for bar, n in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, n + 6, f'{n:,}\n{n / counts.sum() * 100:.0f}%',
                ha='center', va='bottom', fontsize=7, color='#333333', linespacing=1.1)
ax.set_title('DeepDRiD-CFP: images per DR grade', fontsize=11, fontweight='bold', pad=8)
ax.set_xticks(grades); ax.set_xticklabels(GRADE_NAMES, fontsize=8.5)
ax.set_xlabel('DR grade (per eye)'); ax.set_ylabel('Number of images')
ax.set_ylim(0, manifest['grade'].value_counts().max() * 0.9)
ax.legend(frameon=False, fontsize=9)
plt.show()

## 4 · Single-Run Helper

`src/utils/experiment.py` does the work, the same for every notebook: validation every epoch,
`best.ckpt` (best `val_qwk`) and `last.ckpt` after every epoch, resume from `last.ckpt`, test with the
best checkpoint, k = 0 tests the untrained model. Runs land in `results_shared/<config hash>/`, so the
same configuration is never trained twice, whichever notebook asks for it.

Progress: the outer bar counts runs; below it one bar follows the current run epoch by epoch (loss,
validation QWK, best validation QWK) and one follows the batches of the current epoch.

In [ ]:
def spec_overrides(spec: dict) -> list[str]:
    own = [] if spec.get('own_budget', False) else [f'train.epochs={EPOCHS}']
    return own + [f'{k}={v}' for k, v in spec.get('extra', {}).items()]


def run_experiment(name: str, spec: dict, k_shot: int, seed: int, force_retrain: bool = False) -> dict:
    return _run_experiment(name, spec['config'], k_shot, seed, SHARED_RESULTS,
                           overrides=spec_overrides(spec), data_dir=DATA_DIR, force_retrain=force_retrain)


def spec_dir(name: str, k_shot: int, seed: int, variants: dict | None = None):
    # Folder of one run in the shared cache (checkpoints, test_stats.json, geometry.json)
    spec = (variants or {**METHODS, **ABLATIONS})[name]
    return run_dir(SHARED_RESULTS, load_run_config(spec['config'], k_shot, seed, spec_overrides(spec), DATA_DIR))


def run_grid(variants: dict, k_shots: list[int], seeds: list[int], desc: str, train: bool = True) -> pd.DataFrame:
    # Rows of metrics for every (variant, k, seed); train=False only collects runs already in the cache
    rows, missing = [], 0
    pbar = tqdm(total=len(variants) * len(k_shots) * len(seeds), desc=desc, unit='run', dynamic_ncols=True)
    for k in k_shots:
        for name, spec in variants.items():
            for seed in seeds:
                pbar.set_postfix({'variant': name, 'k': k, 'seed': seed})
                if train:
                    m = run_experiment(name, spec, k, seed)
                else:
                    m = cached_metrics(spec['config'], k, seed, SHARED_RESULTS, spec_overrides(spec), DATA_DIR)
                if m is None:
                    missing += 1
                else:
                    rows.append({'Variant': name, 'k': k, 'Seed': seed, **m})
                pbar.update(1)
    pbar.close()
    if missing:
        print(f'{missing} runs not trained yet (set the Phase 2 switch to train them)')
    return pd.DataFrame(rows)

print('run_experiment(), spec_dir() and run_grid() defined.')

## 5 · Phase 1 — Main Model Only

Trains only this notebook's main model over every k and seed. The main notebook and the other
direction notebooks do the same for their main model, all into the shared cache.

In [ ]:
RUN_PHASE1 = True
df_phase1 = run_grid({MAIN: METHODS[MAIN]}, K_SHOTS, SEEDS, f'Phase 1: {MAIN}', train=RUN_PHASE1)
print('Phase 1 results:', df_phase1.shape)

### 5b · Compare the Main Models (master and Directions 1–3)

Reads finished runs from the shared cache by name; trains nothing. A model appears once its own
notebook has run its Phase 1.

In [ ]:
MAIN_MODELS = {                      # display name -> names the run may be recorded under
    'Ours (master)': ['Ours (Full)', 'Ours (master)'],
    'Direction 1':   ['Direction 1'],
    'Direction 2':   ['Direction 2'],
    'Direction 3':   ['Direction 3'],
}
frames = [find_runs(SHARED_RESULTS, names).assign(Model=label) for label, names in MAIN_MODELS.items()]
frames = [f for f in frames if not f.empty and 'qwk' in f]
if frames:
    df_mains = pd.concat(frames, ignore_index=True)
    for m in ('qwk', 'mae', 'odc_full', 'ece', 'nll'):
        if m in df_mains.columns:
            t = df_mains.groupby(['Model', 'k'])[m].mean().unstack('k').reindex(list(MAIN_MODELS)).dropna(how='all')
            print(f'\n{m} (mean over seeds)'); display(t.round(3))
    print('Not run yet:', [m for m in MAIN_MODELS if m not in set(df_mains['Model'])] or 'none')
else:
    print('No main-model runs in the shared cache yet.')

## 6 · Phase 2 — Baselines and Ablations

Set `RUN_PHASE2 = True` when this direction is the best main model in 5b. With it off, the cells below
still collect whatever is already in the shared cache (for example baselines trained in another
notebook), so the tables and figures fill in as runs become available.

In [ ]:
RUN_PHASE2 = False
df_main = run_grid(METHODS, K_SHOTS, SEEDS, 'Phase 2: methods', train=RUN_PHASE2)
df_abl = run_grid(ABLATIONS, ABLATION_K_SHOTS, SEEDS, 'Phase 2: ablations', train=RUN_PHASE2)
print('Methods:', df_main.shape, '| ablations:', df_abl.shape)

## 7 · Results Tables (mean ± std over seeds)

In [ ]:
METRICS = ['qwk', 'mae', 'f1', 'auc', 'acc', 'odc_full', 'odc_offdiag', 'ece']
METRIC_LABELS = {'qwk': 'QWK ↑', 'mae': 'MAE ↓', 'f1': 'Macro-F1 ↑', 'auc': 'AUC ↑', 'acc': 'Accuracy ↑',
                 'odc_full': 'ODC (full) ↑', 'odc_offdiag': 'ODC (off-diag.) ↑', 'ece': 'ECE ↓'}

def aggregate(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    rows = []
    for keys, grp in df.groupby(group_cols, sort=False):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))
        for m in METRICS:
            if m in grp.columns:
                row[f'{m}_mean'] = grp[m].mean()
                row[f'{m}_std'] = grp[m].std(ddof=1) if len(grp) > 1 else 0.0
                row[m] = f"{row[f'{m}_mean']:.3f} ± {row[f'{m}_std']:.3f}"
        rows.append(row)
    return pd.DataFrame(rows)

def k_label(k):
    return {-1: 'full', 0: 'zero-shot'}.get(k, f'{k}-shot')

if not df_main.empty:
    agg_main = aggregate(df_main, ['Variant', 'k'])
    agg_main.to_csv(RESULTS / 'main_results.csv', index=False)
    qwk_table = agg_main.pivot(index='Variant', columns='k', values='qwk').reindex(list(METHODS))
    qwk_table.columns = [k_label(k) for k in qwk_table.columns]
    print('QWK — main comparison'); display(qwk_table)

if not df_abl.empty:
    agg_abl = aggregate(df_abl, ['Variant', 'k'])
    agg_abl.to_csv(RESULTS / 'ablation_results.csv', index=False)
    abl_table = agg_abl.pivot(index='Variant', columns='k', values='qwk').reindex(list(ABLATIONS))
    abl_table.columns = [k_label(k) for k in abl_table.columns]
    print('QWK — Direction 1 ablations'); display(abl_table)

## 8 · Figures

Style matches the main notebook (DejaVu Sans, no top/right spines). Method colours follow a
fixed order that passes colour-blind checks; each method also has its own marker.

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

# Fixed colour + marker per method (colour follows the method, never its rank)
STYLE = {
    'Direction 1':           ('#2a78d6', 'o'),
    'Ours (master)':         ('#eb6834', 's'),
    'OrdinalCLIP (matched)': ('#1baf7a', '^'),
    'CLIP-DR (matched)':     ('#eda100', 'D'),
    'L2RCLIP (our impl.)':   ('#e87ba4', 'v'),
    'CLAP (matched)':        ('#008300', 'P'),
}
CURVE_METHODS = list(STYLE)

### 8a · k-Shot Curves (few-shot points; full data as dashed lines)

In [ ]:
PLOT_METRICS = ['qwk', 'mae', 'odc_full', 'ece']
FEW_SHOT_K = [k for k in K_SHOTS if k >= 0]

if not df_main.empty:
    fig, axes = plt.subplots(1, len(PLOT_METRICS), figsize=(4.8 * len(PLOT_METRICS), 4.2), constrained_layout=True)
    for ax, metric in zip(axes, PLOT_METRICS):
        for name in CURVE_METHODS:
            color, marker = STYLE[name]
            sub = agg_main[(agg_main['Variant'] == name) & (agg_main['k'] >= 0)].sort_values('k')
            if sub.empty:
                continue
            ax.errorbar(sub['k'], sub[f'{metric}_mean'], yerr=sub[f'{metric}_std'], color=color, marker=marker,
                        linewidth=2, markersize=6, capsize=3, elinewidth=1.2, label=name)
            full = agg_main[(agg_main['Variant'] == name) & (agg_main['k'] == -1)]
            if not full.empty:
                ax.axhline(float(full[f'{metric}_mean'].iloc[0]), color=color, linestyle='--', linewidth=1.3, alpha=0.9)
        ax.set_title(METRIC_LABELS[metric], fontsize=12, fontweight='bold')
        ax.set_xlabel('n-Shot')
        ax.set_xticks(FEW_SHOT_K)
        ax.set_xticklabels([str(k) if k > 0 else '0\n(ZS)' for k in FEW_SHOT_K], fontsize=8)
        ax.set_xlim(-1.5, max(FEW_SHOT_K) + 1.5)
    handles, labels = axes[0].get_legend_handles_labels()
    handles.append(plt.Line2D([], [], color='#555555', linestyle='--', linewidth=1.3))
    labels.append('full data (dashed)')
    fig.legend(handles, labels, loc='outside lower center', ncol=len(labels), frameon=False, fontsize=9)
    plt.savefig(FIGURES / 'kshot_curves_d1.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES / 'kshot_curves_d1.png', bbox_inches='tight', dpi=300)
    plt.show()

### 8b · Ablation Bars (QWK at one k)

In [ ]:
ABLATION_K_PLOT = 8

if not df_abl.empty:
    sub = agg_abl[agg_abl['k'] == ABLATION_K_PLOT].set_index('Variant').reindex(list(ABLATIONS)).dropna(subset=['qwk_mean'])
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(sub) + 1.2), constrained_layout=True)
    y = np.arange(len(sub))[::-1]
    colors = ['#2a78d6' if v == 'Direction 1' else '#aab2bd' for v in sub.index]
    ax.barh(y, sub['qwk_mean'], xerr=sub['qwk_std'], color=colors, height=0.6,
            error_kw={'elinewidth': 1.2, 'capsize': 3, 'ecolor': '#333333'})
    for yi, (m, s) in zip(y, zip(sub['qwk_mean'], sub['qwk_std'])):
        ax.text(m + s + 0.01, yi, f'{m:.3f}', va='center', fontsize=8.5, color='#333333')
    if 'Direction 1' in sub.index:
        ax.axvline(sub.loc['Direction 1', 'qwk_mean'], color='#2a78d6', linestyle='--', linewidth=1.2)
    ax.set_yticks(y); ax.set_yticklabels(sub.index, fontsize=9)
    ax.set_xlim(0, 1); ax.set_xlabel('QWK ↑')
    ax.set_title(f'Direction 1 ablations — DeepDRiD-CFP, k = {ABLATION_K_PLOT}', fontsize=11, fontweight='bold')
    plt.savefig(FIGURES / f'ablation_d1_k{ABLATION_K_PLOT}.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES / f'ablation_d1_k{ABLATION_K_PLOT}.png', bbox_inches='tight', dpi=300)
    plt.show()

### 8c · Normalized Confusion Matrices (summed over seeds)

In [ ]:
CM_K = 8
CM_METHODS = ['Direction 1', 'OrdinalCLIP (matched)', 'CLIP-DR (matched)']

def summed_confusion(name, k):
    rows = df_main[(df_main['Variant'] == name) & (df_main['k'] == k)]
    if rows.empty:
        return None
    cm = np.sum([np.array(c) for c in rows['confusion_matrix']], axis=0).astype(float)
    return cm / cm.sum(axis=1, keepdims=True).clip(min=1)

if not df_main.empty:
    mats = [(n, summed_confusion(n, CM_K)) for n in CM_METHODS]
    mats = [(n, m) for n, m in mats if m is not None]
    fig, axes = plt.subplots(1, len(mats), figsize=(4.3 * len(mats), 4.0), constrained_layout=True, squeeze=False)
    for ax, (name, cm) in zip(axes[0], mats):
        im = ax.imshow(cm, cmap='Reds', vmin=0, vmax=1)
        for i in range(5):
            for j in range(5):
                ax.text(j, i, f'{cm[i, j]:.3f}', ha='center', va='center', fontsize=8,
                        color='white' if cm[i, j] > 0.55 else '#222222')
        ax.set_xticks(range(5)); ax.set_yticks(range(5))
        ax.set_xticklabels([f'Grade {g}' for g in range(5)], fontsize=8, rotation=30)
        ax.set_yticklabels([f'Grade {g}' for g in range(5)], fontsize=8)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(name, fontsize=11, fontweight='bold')
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
    fig.suptitle(f'Normalized confusion matrices — k = {CM_K}', fontsize=12, fontweight='bold')
    plt.savefig(FIGURES / f'confusion_d1_k{CM_K}.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES / f'confusion_d1_k{CM_K}.png', bbox_inches='tight', dpi=300)
    plt.show()

### 8d · Error Decay with Ordinal Distance

Share of each true grade's predictions that land |pred − true| = Δ grades away, averaged over
grades and seeds. An ordinal model puts most mass at Δ = 0 and less at every further step.

In [ ]:
DECAY_K = 8

if not df_main.empty:
    fig, ax = plt.subplots(figsize=(7.2, 3.8), constrained_layout=True)
    names = [n for n in CURVE_METHODS if ((df_main['Variant'] == n) & (df_main['k'] == DECAY_K)).any()]
    width = 0.8 / max(1, len(names))
    for i, name in enumerate(names):
        mass = np.mean([np.array(m) for m in df_main[(df_main['Variant'] == name) & (df_main['k'] == DECAY_K)]['mean_mass_by_dist']], axis=0)
        ax.bar(np.arange(5) + (i - (len(names) - 1) / 2) * width, mass, width=width * 0.92,
               color=STYLE[name][0], label=name)
    ax.set_xticks(range(5)); ax.set_xticklabels([f'Δ = {d}' for d in range(5)])
    ax.set_ylabel('Share of predictions'); ax.set_ylim(0, 1)
    ax.set_title(f'Prediction mass by ordinal distance — k = {DECAY_K}', fontsize=11, fontweight='bold')
    ax.legend(frameon=False, fontsize=8.5, ncol=2)
    plt.savefig(FIGURES / f'error_decay_d1_k{DECAY_K}.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES / f'error_decay_d1_k{DECAY_K}.png', bbox_inches='tight', dpi=300)
    plt.show()

### 8e · Grade-Embedding Geometry During Training (Direction 1)

From `geometry.json`: the angle between consecutive steps of the grade path (smaller = straighter,
more ordered path), the share of satisfied ladder constraints, and the final pairwise distances.

In [ ]:
GEO_K, GEO_SEED = 8, 1
geo_file = spec_dir('Direction 1', GEO_K, GEO_SEED) / 'geometry.json'

if geo_file.exists():
    geo = json.load(open(geo_file))
    epochs = [g['epoch'] for g in geo]
    angles = np.array([g['step_angles_deg'] for g in geo])
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
    for j in range(angles.shape[1]):
        axes[0].plot(epochs, angles[:, j], linewidth=1.8, label=f'step {j}→{j + 1} vs {j + 1}→{j + 2}')
    axes[0].set_title('Turning angle along the grade path', fontweight='bold'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Degrees'); axes[0].legend(frameon=False, fontsize=8)
    axes[1].plot(epochs, [g['ladder_satisfied_frac'] for g in geo], color='#2a78d6', linewidth=2)
    axes[1].set_ylim(0, 1.05); axes[1].set_xlabel('Epoch')
    axes[1].set_title('Ladder constraints satisfied', fontweight='bold')
    dist = np.array(geo[-1]['distance_matrix'])
    im = axes[2].imshow(dist, cmap='Blues')
    for i in range(5):
        for j in range(5):
            axes[2].text(j, i, f'{dist[i, j]:.3f}', ha='center', va='center', fontsize=7.5,
                         color='white' if dist[i, j] > dist.max() * 0.6 else '#222222')
    axes[2].set_xticks(range(5)); axes[2].set_yticks(range(5))
    axes[2].set_title('Cosine distance between grades (last epoch)', fontweight='bold')
    fig.colorbar(im, ax=axes[2], shrink=0.8)
    plt.savefig(FIGURES / f'geometry_d1_k{GEO_K}_s{GEO_SEED}.pdf', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print(f'No geometry file yet: {geo_file}')